# 부서 연결 Retriever 프로젝트 v2

## 변경 목적

1. GPU/Colab 미사용, 일반 PC Jupyter Notebook 기준
2. `업무담당규정.docx`는 **검색용 chunk로 사용하지 않고**, 1차 도메인 적응 학습용 문서로만 사용
3. `dept.xlsx`는 `HNG_BR_NM`, `OFWRK_NM` 컬럼 기준으로 로드
4. `dept.xlsx`는 **부서별 5행 단위 chunk**로 분할하여 최종 검색 corpus로 사용
5. SentenceTransformer 학습은 2회로 분리
   - 1차: `업무담당규정.docx` 기반 도메인 적응
   - 2차: `dept.xlsx` 라벨 기반 부서 연결 학습
6. query 입력 시 `HNG_BR_NM` 상위 5개 출력 후, 다수결 방식으로 최종 1건 출력


In [1]:


# !pip install -U sentence-transformers rank_bm25 python-docx openpyxl tqdm scikit-learn
# !pip install rank_bm25
# !pip install -U datasets

In [2]:
# ============================================================
# 1. 라이브러리 및 기본 설정
# ============================================================

import os
import re
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

import torch

from datasets import Dataset 

# CPU 고정
DEVICE = "cpu"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("DEVICE:", DEVICE)
print("torch:", torch.__version__)


DEVICE: cpu
torch: 2.12.0+cpu


C:\Users\User\AppData\Local\Temp\ipykernel_3356\1324928047.py:18: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [3]:
# ============================================================
# 2. 파일 경로 및 학습 옵션
# ============================================================

BASE_DOC_PATH = "./업무담당규정.docx"
DEPT_XLSX_PATH = "./dept.xlsx"

# 기본 SentenceTransformer 모델
EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"

# 2단계 학습 결과 저장 경로
STAGE1_MODEL_DIR = "./stage1_domain_retriever"
STAGE2_MODEL_DIR = "./stage2_dept_retriever"

# 테스트 목적이므로 기본 True
# CPU에서 오래 걸리면 False로 바꾸고 기존 저장 모델을 로드하세요.
RUN_STAGE1_DOMAIN_TRAIN = True
RUN_STAGE2_DEPT_TRAIN = True

# 1차 도메인 학습 설정
STAGE1_EPOCHS = 1
STAGE1_BATCH_SIZE = 8
STAGE1_LR = 2e-5
STAGE1_MAX_TRAIN_TEXTS = 1000   # CPU 보호용. 전체 문단을 쓰고 싶으면 None

# 2차 dept 학습 설정
STAGE2_EPOCHS = 1
STAGE2_BATCH_SIZE = 8
STAGE2_LR = 2e-5

# dept.xlsx chunk 규칙
DEPT_CHUNK_SIZE = 5

# 검색 설정
DENSE_WEIGHT = 0.55
TFIDF_WEIGHT = 0.25
BM25_WEIGHT = 0.20

print("BASE_DOC_PATH:", BASE_DOC_PATH)
print("DEPT_XLSX_PATH:", DEPT_XLSX_PATH)


BASE_DOC_PATH: ./업무담당규정.docx
DEPT_XLSX_PATH: ./dept.xlsx


In [4]:
# ============================================================
# 3. dept.xlsx 로드 및 검증
# ============================================================
# 필수 컬럼:
# - HNG_BR_NM : 부서명
# - OFWRK_NM  : 업무 설명 / 역할 설명

def clean_text(x) -> str:
    x = "" if pd.isna(x) else str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def load_dept_excel(path: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"dept.xlsx 파일을 찾을 수 없습니다: {path.resolve()}")

    df = pd.read_excel(path)

    required_cols = ["HNG_BR_NM", "OFWRK_NM"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"dept.xlsx에 필요한 컬럼이 없습니다: {missing}\n"
            f"현재 컬럼: {list(df.columns)}"
        )

    df = df[required_cols].copy()
    df["HNG_BR_NM"] = df["HNG_BR_NM"].apply(clean_text)
    df["OFWRK_NM"] = df["OFWRK_NM"].apply(clean_text)

    df = df.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    df = df.dropna(subset=["HNG_BR_NM", "OFWRK_NM"]).reset_index(drop=True)

    # 원본 행번호 추적용
    df["orig_row_id"] = np.arange(len(df))

    return df


dept_df = load_dept_excel(DEPT_XLSX_PATH)

print("dept_df shape:", dept_df.shape)
display(dept_df.head())
print("부서 수:", dept_df["HNG_BR_NM"].nunique())


dept_df shape: (2544, 3)


,HNG_BR_NM,OFWRK_NM,orig_row_id
0,디지털 HR부,"[Cell장 담당업무]디지털/ICT 인사 운용, 채용, 연수 전반사항 [담당] AX...",0
1,디지털 HR부,"디지털/ICT 연수 기획/운영, 신입직원 연수/온보딩",1
2,디지털 HR부,"디지털/ICT 연수 기획/운영, 신입직원 채용/연수/온보딩",2
3,디지털 HR부,"디지털/ICT 직원 인사 [담당] Tech그룹, 정보보호본부, 기관·제휴개발부",3
4,디지털 HR부,"디지털/ICT 채용, 직무 및 CDP, 신입직원 연수/온보딩",4


부서 수: 103


In [5]:
# ============================================================
# 4. 업무담당규정.docx 로드
# ============================================================

def read_docx_paragraphs(docx_path: str, min_chars: int = 20) -> list:
    from docx import Document

    path = Path(docx_path)
    if not path.exists():
        raise FileNotFoundError(f"업무담당규정.docx 파일을 찾을 수 없습니다: {path.resolve()}")

    doc = Document(str(path))
    paragraphs = []

    for p in doc.paragraphs:
        t = clean_text(p.text)
        if len(t) >= min_chars:
            paragraphs.append(t)

    # 표 안의 텍스트도 추가
    for table in doc.tables:
        for row in table.rows:
            row_texts = []
            for cell in row.cells:
                t = clean_text(cell.text)
                if t:
                    row_texts.append(t)
            merged = " ".join(row_texts)
            if len(merged) >= min_chars:
                paragraphs.append(merged)

    # 중복 제거, 순서 유지
    seen = set()
    unique_paragraphs = []
    for t in paragraphs:
        if t not in seen:
            unique_paragraphs.append(t)
            seen.add(t)

    return unique_paragraphs


domain_texts = read_docx_paragraphs(BASE_DOC_PATH, min_chars=20)

print("업무담당규정 도메인 학습 문단 수:", len(domain_texts))
print("\n[샘플]")
for t in domain_texts[:5]:
    print("-", t[:200])


업무담당규정 도메인 학습 문단 수: 2114

[샘플]
- 제1조(목적) 이 규정은 본부조직 및 영업조직의 담당업무를 정하는 데 그 목적이 있다. 단, 각 부서는 본 규정의 담당업무에도 불구하고, 은행의 중장기적인 가치 향상을 위한 부서간 협업에 역량과 자원을 우선 배분해야 하며, 별도의 명시 및 외규, 내규의 제약이 없는 한 본 규정의 효력 범위를 국내에 제한하지 않는다.
- 제2조(기안) 이 규정의 제정, 개폐에 기안권자는 종합기획부장으로 한다. 단, 다음과 같은 사항과 관련된 기안권자는 소관부서장으로 하되 종합기획부장의 합의를 거치도록 한다.
- 1. 동일 그룹 또는 본부 내 부서간 업무조정에 따른 개정
- 2. 타 그룹 또는 본부에 영향을 미치지 않는 담당업무의 추가, 신규에 따른 개정
- 3. 은행장이 별도로 결정한 사항에 따른 부수적인 개정


In [6]:
# ============================================================
# 5. 1차 학습: 업무담당규정.docx 기반 도메인 지식화 및 전처리
# ============================================================


def prepare_stage1_domain_examples(domain_texts: list, max_train_texts=1000) -> list:
    texts = [clean_text(t) for t in domain_texts if len(clean_text(t)) >= 20]

    # 너무 긴 문단은 CPU 학습 안정성을 위해 자름
    texts = [t[:512] for t in texts]

    # 중복 제거
    texts = list(dict.fromkeys(texts))

    if max_train_texts is not None and len(texts) > max_train_texts:
        random.shuffle(texts)
        texts = texts[:max_train_texts]

    examples = [InputExample(texts=[t, t]) for t in texts]
    return examples


def train_stage1_domain_model(
    base_model_name: str,
    domain_texts: list,
    output_dir: str,
    epochs: int = 1,
    batch_size: int = 8,
    lr: float = 2e-5,
    max_train_texts=1000
):
    train_examples = prepare_stage1_domain_examples(
        domain_texts=domain_texts,
        max_train_texts=max_train_texts
    )

    if len(train_examples) == 0:
        raise ValueError("1차 학습용 domain_texts가 비어 있습니다.")

    print("1차 학습 예제 수:", len(train_examples))

    model = SentenceTransformer(base_model_name, device=DEVICE)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    warmup_steps = max(1, int(len(train_loader) * 0.1))

    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": lr},
        show_progress_bar=True
    )

    model.save(output_dir)
    print("1차 도메인 모델 저장 완료:", output_dir)
    return model


if RUN_STAGE1_DOMAIN_TRAIN:
    stage1_model = train_stage1_domain_model(
        base_model_name=EMBEDDING_MODEL_NAME,
        domain_texts=domain_texts,
        output_dir=STAGE1_MODEL_DIR,
        epochs=STAGE1_EPOCHS,
        batch_size=STAGE1_BATCH_SIZE,
        lr=STAGE1_LR,
        max_train_texts=STAGE1_MAX_TRAIN_TEXTS
    )
    STAGE1_BASE_FOR_STAGE2 = STAGE1_MODEL_DIR
else:
    if Path(STAGE1_MODEL_DIR).exists():
        print("기존 1차 모델 사용:", STAGE1_MODEL_DIR)
        STAGE1_BASE_FOR_STAGE2 = STAGE1_MODEL_DIR
    else:
        print("1차 학습 생략. 기본 모델 사용:", EMBEDDING_MODEL_NAME)
        STAGE1_BASE_FOR_STAGE2 = EMBEDDING_MODEL_NAME


1차 학습 예제 수: 1000


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


1차 도메인 모델 저장 완료: ./stage1_domain_retriever


In [7]:
# ============================================================
# 6. dept.xlsx 5행 chunk 생성
# ============================================================

def build_dept_chunks_by_department(dept_df: pd.DataFrame, chunk_size: int = 5) -> pd.DataFrame:
    rows = []
    chunk_no = 0

    for dept, g in dept_df.groupby("HNG_BR_NM", sort=False):
        g = g.sort_values("orig_row_id").reset_index(drop=True)

        for start in range(0, len(g), chunk_size):
            part = g.iloc[start:start + chunk_size].copy()
            works = part["OFWRK_NM"].astype(str).tolist()

            text = (
                f"[부서명] {dept}\n"
                f"[담당업무]\n" +
                "\n".join([f"- {w}" for w in works])
            )

            rows.append({
                "source": "dept.xlsx",
                "chunk_id": f"dept_chunk_{chunk_no}",
                "HNG_BR_NM": dept,
                "text": text,
                "work_items": works,
                "n_rows": len(part),
                "orig_row_start": int(part["orig_row_id"].min()),
                "orig_row_end": int(part["orig_row_id"].max())
            })

            chunk_no += 1

    return pd.DataFrame(rows)


dept_corpus = build_dept_chunks_by_department(dept_df, chunk_size=DEPT_CHUNK_SIZE)

# 이번 구조에서 최종 검색 corpus는 dept.xlsx chunk만 사용합니다.
# 업무담당규정.docx는 1차 학습에만 사용하고, 검색 대상에서는 제외합니다.
corpus_df = dept_corpus[[
    "source", "chunk_id", "HNG_BR_NM", "text", "work_items", "n_rows", "orig_row_start", "orig_row_end"
]].copy()

print("dept chunk corpus shape:", corpus_df.shape)
print("부서 수:", corpus_df["HNG_BR_NM"].nunique())
display(corpus_df.head(10))


dept chunk corpus shape: (559, 8)
부서 수: 103


,source,chunk_id,HNG_BR_NM,text,work_items,n_rows,orig_row_start,orig_row_end
0,dept.xlsx,dept_chunk_0,디지털 HR부,[부서명] 디지털 HR부\n[담당업무]\n- [Cell장 담당업무]디지털/ICT 인...,"[[Cell장 담당업무]디지털/ICT 인사 운용, 채용, 연수 전반사항 [담당] A...",5,0,4
1,dept.xlsx,dept_chunk_1,디지털 HR부,"[부서명] 디지털 HR부\n[담당업무]\n- 디지털/ICT 채용, 직무 및 CDP,...","[디지털/ICT 채용, 직무 및 CDP, 인사제도 운영(부), 민원소통담당자(부),...",2,5,6
2,dept.xlsx,dept_chunk_2,기관영업2부,[부서명] 기관영업2부\n[담당업무]\n- ○ 공공기관 주거래협약 관리 ○ 공공기관...,"[○ 공공기관 주거래협약 관리 ○ 공공기관 계약서 관리 담당자, ○ 공공기관(자금)...",5,7,11
3,dept.xlsx,dept_chunk_3,기관영업2부,[부서명] 기관영업2부\n[담당업무]\n- ○ 대학교/병원 신규유치 및 고객관리 ○...,[○ 대학교/병원 신규유치 및 고객관리 ○ 대학교/병원 만기/재협약 관리 ○ 대학교...,5,12,16
4,dept.xlsx,dept_chunk_4,기관영업2부,[부서명] 기관영업2부\n[담당업무]\n- ○ 대학교/병원 신규유치 및 고객관리 ○...,[○ 대학교/병원 신규유치 및 고객관리 ○ 대학교/병원 만기/재협약 관리 ○ 대학교...,5,17,21
5,dept.xlsx,dept_chunk_5,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- - OCR/비OCR 집계대사 및 물류발송...,"[- OCR/비OCR 집계대사 및 물류발송, AI_OCR 수기고지서 데이터화 작업,...",5,22,26
6,dept.xlsx,dept_chunk_6,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- ○ 나라사랑카드 사업 관리 ○ 국군 연계...,"[○ 나라사랑카드 사업 관리 ○ 국군 연계 마케팅(정), ○ 사업운영(정) ○ 나라...",5,27,31
7,dept.xlsx,dept_chunk_7,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- ○ 사업이행 (정) ○ 병무청 발급소 운...,"[○ 사업이행 (정) ○ 병무청 발급소 운영 ○ 유관기관 소통(국방부, 병무청, 군...",5,32,36
8,dept.xlsx,dept_chunk_8,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- 군 현장영업\n- 사업계획(정) 재무 및...,"[군 현장영업, 사업계획(정) 재무 및 예산관리(정) KPI 관리(정) 부서 수신문...",5,37,41
9,dept.xlsx,dept_chunk_9,기관영업1부,"[부서명] 기관영업1부\n[담당업무]\n- 시도금고운영CELL 총괄, 세입(시스템)...","[시도금고운영CELL 총괄, 세입(시스템) 지자체 금고 시스템 관리_시스템 운영 이...",5,42,46


In [8]:
# ============================================================
# 7. 2차 학습: dept.xlsx 라벨 기반 부서 연결 학습
# ============================================================
# 학습 데이터:
# - query 역할: OFWRK_NM 개별 업무 문장
# - positive context 역할: 해당 업무가 들어있는 5행 chunk
#

def prepare_stage2_dept_examples(corpus_df: pd.DataFrame) -> list:
    examples = []

    for _, row in corpus_df.iterrows():
        dept = str(row["HNG_BR_NM"])
        chunk_text = str(row["text"])
        work_items = row["work_items"]

        # 1) 개별 업무 문장 -> 해당 5행 chunk
        for w in work_items:
            w = clean_text(w)
            if len(w) >= 3:
                examples.append(InputExample(texts=[w, chunk_text]))

                # 부서명이 포함된 표현도 추가
                examples.append(InputExample(texts=[f"{dept} {w}", chunk_text]))

        # 2) 부서명 자체 -> 해당 chunk
        examples.append(InputExample(texts=[dept, chunk_text]))

    random.shuffle(examples)
    return examples


def train_stage2_dept_model(
    stage1_or_base_model_name: str,
    corpus_df: pd.DataFrame,
    output_dir: str,
    epochs: int = 1,
    batch_size: int = 8,
    lr: float = 2e-5
):
    train_examples = prepare_stage2_dept_examples(corpus_df)

    if len(train_examples) == 0:
        raise ValueError("2차 학습용 dept 예제가 비어 있습니다.")

    print("2차 학습 예제 수:", len(train_examples))

    model = SentenceTransformer(stage1_or_base_model_name, device=DEVICE)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    warmup_steps = max(1, int(len(train_loader) * 0.1))

    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": lr},
        show_progress_bar=True
    )

    model.save(output_dir)
    print("2차 dept 모델 저장 완료:", output_dir)
    return model


if RUN_STAGE2_DEPT_TRAIN:
    embedder = train_stage2_dept_model(
        stage1_or_base_model_name=STAGE1_BASE_FOR_STAGE2,
        corpus_df=corpus_df,
        output_dir=STAGE2_MODEL_DIR,
        epochs=STAGE2_EPOCHS,
        batch_size=STAGE2_BATCH_SIZE,
        lr=STAGE2_LR
    )
else:
    if Path(STAGE2_MODEL_DIR).exists():
        print("기존 2차 모델 사용:", STAGE2_MODEL_DIR)
        embedder = SentenceTransformer(STAGE2_MODEL_DIR, device=DEVICE)
    else:
        print("2차 학습 생략. 1차 또는 기본 모델 사용:", STAGE1_BASE_FOR_STAGE2)
        embedder = SentenceTransformer(STAGE1_BASE_FOR_STAGE2, device=DEVICE)

print("최종 embedder 준비 완료")


2차 학습 예제 수: 5639


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 8. 임베딩 / TF-IDF / BM25 인덱스 생성
# ============================================================

def simple_tokenize(text: str) -> list:
    return re.findall(r"[가-힣A-Za-z0-9]+", str(text).lower())


def normalize_matrix(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)


def top_indices(scores: np.ndarray, k: int) -> np.ndarray:
    k = min(k, len(scores))
    return np.argsort(scores)[-k:][::-1]


corpus_texts = corpus_df["text"].astype(str).tolist()

# 1) Dense embedding
corpus_embs = embedder.encode(
    corpus_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=64
)
corpus_embs = normalize_matrix(corpus_embs)

# 2) TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_texts)

# 3) BM25
tokenized_corpus = [simple_tokenize(t) for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Dense embeddings:", corpus_embs.shape)
print("TF-IDF matrix:", tfidf_matrix.shape)
print("BM25 corpus:", len(tokenized_corpus))


In [ ]:
# ============================================================
# 9. Query → HNG_BR_NM 상위 5개 + 다수결 최종 1건
# ============================================================

def minmax_norm(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x
    return (x - x.min()) / (np.ptp(x) + 1e-12)


def retrieve_scored_rows(
    query: str,
    candidate_k: int = 80,
    dense_weight: float = DENSE_WEIGHT,
    tfidf_weight: float = TFIDF_WEIGHT,
    bm25_weight: float = BM25_WEIGHT
) -> pd.DataFrame:
    """
    dense / TF-IDF / BM25를 각각 계산한 뒤,
    각 방식의 상위 후보를 union하여 최종 점수를 계산합니다.

    기존처럼 dense top-k 안에서만 lexical을 계산하면,
    dense가 틀렸을 때 TF-IDF/BM25가 구조적으로 구제하지 못합니다.
    그래서 세 방식의 후보를 합칩니다.
    """
    query = clean_text(query)
    if not query:
        raise ValueError("query가 비어 있습니다.")

    candidate_k = min(candidate_k, len(corpus_df))

    # 1) Dense cosine similarity
    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-12)
    dense_scores_all = corpus_embs @ q_emb[0]

    # 2) TF-IDF
    q_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores_all = (tfidf_matrix @ q_tfidf.T).toarray().ravel()

    # 3) BM25
    bm25_scores_all = np.asarray(bm25.get_scores(simple_tokenize(query)))

    # 방식별 후보 union
    dense_idx = top_indices(dense_scores_all, candidate_k)
    tfidf_idx = top_indices(tfidf_scores_all, candidate_k)
    bm25_idx = top_indices(bm25_scores_all, candidate_k)

    candidate_idx = np.unique(np.concatenate([dense_idx, tfidf_idx, bm25_idx]))

    d = dense_scores_all[candidate_idx]
    t = tfidf_scores_all[candidate_idx]
    b = bm25_scores_all[candidate_idx]

    final_scores = (
        dense_weight * minmax_norm(d) +
        tfidf_weight * minmax_norm(t) +
        bm25_weight * minmax_norm(b)
    )

    result = corpus_df.iloc[candidate_idx].copy()
    result["dense_score"] = d
    result["tfidf_score"] = t
    result["bm25_score"] = b
    result["score"] = final_scores

    result = result.sort_values("score", ascending=False).reset_index(drop=True)
    result["rank"] = np.arange(1, len(result) + 1)

    return result


def predict_department(
    query: str,
    top_n_departments: int = 5,
    candidate_k: int = 80,
    vote_pool_size: int = 20
):
    """
    1) query와 유사한 chunk 후보 검색
    2) 상위 vote_pool_size개 chunk를 투표 풀로 사용
    3) HNG_BR_NM 기준 다수결
    4) 동률이면 score_sum, score_max 순으로 정렬
    """
    scored_rows = retrieve_scored_rows(query, candidate_k=candidate_k)

    vote_pool_size = min(vote_pool_size, len(scored_rows))
    vote_pool = scored_rows.head(vote_pool_size).copy()

    # rank가 높을수록 가중치를 크게 주는 보조 지표
    vote_pool["rank_weight"] = 1 / vote_pool["rank"]

    summary = (
        vote_pool
        .groupby("HNG_BR_NM")
        .agg(
            vote_count=("HNG_BR_NM", "size"),
            score_sum=("score", "sum"),
            score_max=("score", "max"),
            rank_weight_sum=("rank_weight", "sum"),
            best_source=("source", "first"),
            best_chunk_id=("chunk_id", "first"),
            best_text=("text", "first")
        )
        .reset_index()
    )

    summary = summary.sort_values(
        ["vote_count", "score_sum", "score_max", "rank_weight_sum"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    top5 = summary.head(top_n_departments).copy()
    final_department = top5.iloc[0]["HNG_BR_NM"] if len(top5) else None

    return {
        "query": query,
        "final_department": final_department,
        "top5_departments": top5,
        "scored_rows": scored_rows,
        "vote_pool": vote_pool
    }


def print_prediction(result: dict):
    print("[입력 Query]")
    print(result["query"])
    print("\n[HNG_BR_NM 상위 5개 - 다수결 기준]")

    top5 = result["top5_departments"]
    if len(top5) == 0:
        print("검색 결과 없음")
        return

    for i, row in top5.iterrows():
        print(
            f"{i+1}. {row['HNG_BR_NM']} "
            f"| votes={int(row['vote_count'])} "
            f"| score_sum={row['score_sum']:.4f} "
            f"| score_max={row['score_max']:.4f}"
        )

    print("\n[최종 출력 1건]")
    print(result["final_department"])


In [ ]:
# ============================================================
# 10. 단건 테스트
# ============================================================


with open("query1.txt", "r", encoding="utf-8") as f:
    query1 = f.read()

result = predict_department(
    query=query1,
    top_n_departments=5,
    candidate_k=80,
    vote_pool_size=20
)

print_prediction(result)

display(result["top5_departments"])
display(result["scored_rows"].head(20)[[
    "rank", "HNG_BR_NM", "source", "chunk_id", "score",
    "dense_score", "tfidf_score", "bm25_score", "text"
]])


In [ ]:

with open("query2.txt", "r", encoding="utf-8") as f:
    query2 = f.read()

    
result = predict_department(
    query=query2,
    top_n_departments=5,
    candidate_k=80,
    vote_pool_size=20
)

print_prediction(result)

display(result["top5_departments"])
display(result["scored_rows"].head(20)[[
    "rank", "HNG_BR_NM", "source", "chunk_id", "score",
    "dense_score", "tfidf_score", "bm25_score", "text"
]])

In [ ]:
# # ============================================================
# # 11. 여러 query 배치 예측
# # ============================================================

# def batch_predict(
#     queries: list,
#     top_n_departments: int = 5,
#     candidate_k: int = 80,
#     vote_pool_size: int = 20
# ) -> pd.DataFrame:
#     rows = []

#     for q in tqdm(queries, desc="predict"):
#         r = predict_department(
#             query=q,
#             top_n_departments=top_n_departments,
#             candidate_k=candidate_k,
#             vote_pool_size=vote_pool_size
#         )

#         top5 = r["top5_departments"]["HNG_BR_NM"].tolist()

#         row = {
#             "query": q,
#             "final_department": r["final_department"]
#         }

#         for i in range(top_n_departments):
#             row[f"top{i+1}"] = top5[i] if i < len(top5) else None

#         rows.append(row)

#     return pd.DataFrame(rows)


# sample_queries = [
#     """""""",
#     """""""",
#     """"""""
# ]

# batch_result = batch_predict(sample_queries)
# display(batch_result)


In [ ]:
# # ============================================================
# # 12. 평가 데이터가 있을 경우 정확도 계산
# # ============================================================
# # 평가 파일 컬럼 예시:
# # - 문장
# # - 유관부서

# TEST_XLSX_PATH = "./test_set.xlsx"

# def evaluate_if_exists(test_path: str):
#     if not Path(test_path).exists():
#         print(f"평가 파일 없음: {test_path}")
#         return None

#     test_df = pd.read_excel(test_path)

#     required = ["문장", "유관부서"]
#     missing = [c for c in required if c not in test_df.columns]
#     if missing:
#         raise ValueError(f"평가 파일에 필요한 컬럼이 없습니다: {missing}")

#     pred_df = batch_predict(test_df["문장"].astype(str).tolist())

#     out = pd.concat(
#         [test_df.reset_index(drop=True), pred_df.drop(columns=["query"])],
#         axis=1
#     )

#     out["correct_top1"] = out["final_department"].astype(str) == out["유관부서"].astype(str)

#     top_cols = ["top1", "top2", "top3", "top4", "top5"]
#     out["correct_top5"] = out.apply(
#         lambda r: str(r["유관부서"]) in [str(r[c]) for c in top_cols],
#         axis=1
#     )

#     print("Top-1 accuracy:", out["correct_top1"].mean())
#     print("Top-5 accuracy:", out["correct_top5"].mean())

#     return out

# # eval_result = evaluate_if_exists(TEST_XLSX_PATH)
# # display(eval_result.head())


## 실행 순서 요약

1. `업무담당규정.docx`, `dept.xlsx`를 노트북과 같은 폴더에 둔다.
2. 위에서부터 순서대로 실행한다.
3. CPU가 느리면 다음 값을 줄인다.
   - `STAGE1_MAX_TRAIN_TEXTS`
   - `STAGE1_EPOCHS`
   - `STAGE2_EPOCHS`
   - `STAGE1_BATCH_SIZE`, `STAGE2_BATCH_SIZE`
4. 학습이 너무 오래 걸리면 1회 학습 후 다음부터는 아래처럼 바꿔 저장 모델을 재사용한다.

```python
RUN_STAGE1_DOMAIN_TRAIN = False
RUN_STAGE2_DEPT_TRAIN = False
```
